# Spark Cluster Playground

This notebook runs locally in Cursor/Jupyter and submits jobs to the Spark cluster from `docker-compose` (`spark://localhost:7077`).

In [1]:
from pathlib import Path
import sys

# Make repo root importable regardless of Jupyter working directory
root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from notebooks.spark_session import create_spark_session

spark = create_spark_session('gba-notebook')
spark


:: loading settings :: url = jar:file:/Users/vadim.sokoltsov/learning/github-batch-analytics/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/vadim.sokoltsov/.ivy2/cache
The jars for the packages stored in: /Users/vadim.sokoltsov/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1851a32b-92dd-428c-88f9-9bfe7ea0f9b9;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 103ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number|

In [2]:
print('master:', spark.sparkContext.master)
print('app:', spark.sparkContext.appName)


master: spark://localhost:7077
app: gba-notebook


26/03/12 23:40:51 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [3]:
spark.range(10).withColumnRenamed("id", "n").show()

+---+
|  n|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



In [4]:
path = "s3a://gba-landing-zone-prod/raw/gharchive/dt=2026-03-08/hr=0/events.json.gz"
df = spark.read.json(path)
df.select("id", "type", "repo.name", "actor.login").show(5, truncate=False)

26/03/12 23:40:55 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/03/12 23:40:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+-----------------------------+------------------------------+--------------+
|id        |type                         |name                          |login         |
+----------+-----------------------------+------------------------------+--------------+
|7181765825|IssuesEvent                  |kubernetes/contributor-site   |k8s-ci-robot  |
|7181765850|IssuesEvent                  |space-wizards/space-station-14|slarticodefast|
|7181765895|PullRequestEvent             |Dasharo/coreboot              |mkopec        |
|7181765898|PullRequestReviewCommentEvent|ChrisRomp/copilot-bridge      |Copilot       |
|7181765903|IssuesEvent                  |cocredo/cocredo-status        |cocredo       |
+----------+-----------------------------+------------------------------+--------------+
only showing top 5 rows



In [5]:
spark.stop()